In [1]:
!cd /kaggle/working && git clone https://github.com/rizmyabdulla/SinhalaTTS.git SinhalaTTS
!cp /kaggle/working/SinhalaTTS/*.py /kaggle/working/SinhalaTTS/*.yaml /kaggle/working/SinhalaTTS/*.sh /kaggle/working/SinhalaTTS/*.json /kaggle/working/ 2>/dev/null; true

Cloning into 'SinhalaTTS'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 56 (delta 28), reused 49 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 52.57 KiB | 2.10 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [2]:
# =============================================================================
# CELL 1 — Environment setup
# =============================================================================
# We expect to run on a fresh Kaggle kernel. Kaggle's base image already
# has Python 3.11 and PyTorch 2.x with CUDA. The trick is to install
# the CosyVoice-specific stack (conformer, pyworld, matcha, etc.) into
# the user site-packages.

import os, sys, subprocess, time, json, shutil, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

WORKDIR = Path("/kaggle/working")
WORKDIR.mkdir(exist_ok=True)
os.chdir(WORKDIR)

try:
    SCRIPT_ROOT = Path(__file__).resolve().parent
except NameError:
    SCRIPT_ROOT = WORKDIR  # notebook cells have no __file__

SCRIPTS_DIR = WORKDIR / "scripts"
CONFIGS_DIR = WORKDIR / "configs"
STUBS_DIR = WORKDIR / "stubs"
_REPO_SCRIPTS = (
    "sinhala_normalize.py",
    "prepare_sinhala_data.py",
    "extract_features.py",
    "build_parquet.py",
    "export_sft_model.py",
    "inference_sinhala.py",
    "train_sinhala_sft.sh",
    "whisper_mel.py",
)
_REPO_STUBS = (
    "stubs/whisper/__init__.py",
    "stubs/whisper/tokenizer.py",
)
_REPO_CONFIGS = (
    "cosyvoice3_sinhala_sft.yaml",
    "ds_stage2.json",
)


def find_repo_file(name: str) -> Path | None:
    """Find a SinhalaTTS repo file under WORKDIR or SCRIPT_ROOT."""
    for root in dict.fromkeys((WORKDIR, SCRIPT_ROOT)):
        path = root / name
        if path.exists():
            return path
    return None


def stage_repo_file(name: str, dest: Path) -> Path:
    """Copy a repo file into dest, refreshing when the source is newer."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    src = find_repo_file(name)
    if src is None:
        if dest.exists():
            return dest
        raise FileNotFoundError(
            f"!! {name} not found under {WORKDIR}. "
            "Copy the SinhalaTTS repo files into /kaggle/working/ "
            "(same folder as this notebook)."
        )
    if src.resolve() != dest.resolve():
        shutil.copy(src, dest)
    return dest


def ensure_repo_layout() -> None:
    """Stage all SinhalaTTS helper scripts and configs under WORKDIR."""
    for name in _REPO_SCRIPTS:
        stage_repo_file(name, SCRIPTS_DIR / name)
    for name in _REPO_STUBS:
        stage_repo_file(name, WORKDIR / name)
    for name in _REPO_CONFIGS:
        stage_repo_file(name, CONFIGS_DIR / name)
    os.chmod(SCRIPTS_DIR / "train_sinhala_sft.sh", 0o755)


def bootstrap_whisper_stub() -> None:
    """Ensure Py3.12-safe whisper stub exists (no openai-whisper pip install)."""
    SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
    (STUBS_DIR / "whisper").mkdir(parents=True, exist_ok=True)
    mel_dest = SCRIPTS_DIR / "whisper_mel.py"
    stub_init = STUBS_DIR / "whisper" / "__init__.py"
    stub_tok = STUBS_DIR / "whisper" / "tokenizer.py"
    if not mel_dest.exists():
        stage_repo_file("whisper_mel.py", mel_dest)
    for repo_name, dest in (
        ("stubs/whisper/__init__.py", stub_init),
        ("stubs/whisper/tokenizer.py", stub_tok),
    ):
        if find_repo_file(repo_name) is not None:
            stage_repo_file(repo_name, dest)
        elif not dest.exists() and repo_name.endswith("__init__.py"):
            stub_init.write_text(
                "from whisper_mel import log_mel_spectrogram\n"
                '__all__ = ["log_mel_spectrogram"]\n',
                encoding="utf-8",
            )


def whisper_stub_paths() -> None:
    """Put librosa-based whisper stub ahead of site-packages (Py3.12 safe)."""
    bootstrap_whisper_stub()
    for path in (str(SCRIPTS_DIR), str(STUBS_DIR)):
        if path not in sys.path:
            sys.path.insert(0, path)


OPENSLR30_TARBALL_URL = "https://www.openslr.org/resources/30/si_lk.tar.gz"
OPENSLR30_LINES_URL = "https://openslr.trmal.net/resources/30/si_lk.lines.txt"
OPENSLR30_MIN_WAVS = 1000  # corpus ships ~1251 utterances


def count_sin_wavs(root: Path) -> int:
    return len(list(root.rglob("sin_*.wav")))


def find_sin_wav_dir(root: Path) -> Path | None:
    """Return the directory holding the most sin_*.wav files under root."""
    best: tuple[int, Path] | None = None
    seen: set[Path] = set()
    for d in (root, root / "si_lk", root / "si_lk" / "wav", root / "wav"):
        if d.is_dir() and d not in seen:
            seen.add(d)
            n = len(list(d.glob("sin_*.wav")))
            if best is None or n > best[0]:
                best = (n, d)
    for d in root.rglob("*"):
        if not d.is_dir() or d in seen:
            continue
        seen.add(d)
        n = len(list(d.glob("sin_*.wav")))
        if n and (best is None or n > best[0]):
            best = (n, d)
    return best[1] if best and best[0] else None


def normalize_openslr30_corpus(workdir: Path) -> tuple[Path, Path]:
    """Move sin_*.wav into si_lk/wav/ (layout prepare_sinhala_data expects)."""
    sinhala_src = workdir / "si_lk"
    sinhala_src.mkdir(parents=True, exist_ok=True)
    target_wav = sinhala_src / "wav"
    found = find_sin_wav_dir(workdir)
    if found is None:
        target_wav.mkdir(parents=True, exist_ok=True)
        return sinhala_src, target_wav
    if found.resolve() != target_wav.resolve():
        print(f"[2]   normalizing wav layout: {found} -> {target_wav}")
        target_wav.mkdir(parents=True, exist_ok=True)
        for wav in found.glob("sin_*.wav"):
            dest = target_wav / wav.name
            if not dest.exists():
                shutil.move(str(wav), str(dest))
    return sinhala_src, target_wav


def ensure_openslr30_corpus(workdir: Path) -> tuple[Path, Path]:
    """Download/extract OpenSLR30 audio + manifest; return (si_lk/, si_lk/wav/)."""
    tarball = workdir / "si_lk.tar.gz"
    n_wavs = count_sin_wavs(workdir)
    if n_wavs < OPENSLR30_MIN_WAVS:
        print(f"[2] downloading OpenSLR30 audio si_lk.tar.gz (~700 MB) ...")
        sys.stdout.flush()
        if not tarball.exists():
            subprocess.check_call([
                "curl", "-L", "-o", str(tarball),
                OPENSLR30_TARBALL_URL,
            ])
        print("[2] extracting si_lk.tar.gz ...")
        sys.stdout.flush()
        subprocess.check_call(["tar", "-xzf", str(tarball), "-C", str(workdir)])
        n_wavs = count_sin_wavs(workdir)
        print(f"[2]   {n_wavs} sin_*.wav found after extract")
    else:
        print(f"[2] OpenSLR30 audio already present ({n_wavs} sin_*.wav)")

    sinhala_src, wav_dir = normalize_openslr30_corpus(workdir)
    lines_path = sinhala_src / "si_lk.lines.txt"
    if not lines_path.exists():
        print("[2] downloading OpenSLR30 manifest si_lk.lines.txt ...")
        sys.stdout.flush()
        subprocess.check_call([
            "curl", "-L", "-o", str(lines_path),
            OPENSLR30_LINES_URL,
        ])
    else:
        print(f"[2] OpenSLR30 manifest already at {lines_path}")

    n_wavs = len(list(wav_dir.glob("sin_*.wav")))
    if n_wavs < OPENSLR30_MIN_WAVS:
        raise FileNotFoundError(
            f"!! expected ~1251 sin_*.wav, found {n_wavs} under {wav_dir}. "
            f"Tarball: {tarball}"
        )
    print(f"[2]   corpus -> {sinhala_src}")
    print(f"[2]   {n_wavs} wavs under {wav_dir}")
    return sinhala_src, wav_dir

# Pin the exact stack CosyVoice3 was developed against. This is critical
# because mismatched versions of conformer / matcha / pyworld will silently
# break the DiT flow decoder and produce garbled audio at inference.
PYTHON = sys.executable
print(f"[1] Python: {sys.version.split()[0]}  executable: {PYTHON}")

# Install everything. -q to keep logs short; --no-deps because Kaggle's
# base image already has torch/torchaudio/cuda; --no-build-isolation
# so we use the system torch for the build.
REQS = [
    "HyperPyYAML==1.2.3",
    "conformer==0.3.2",
    "diffusers==0.29.0",
    "hydra-core==1.3.2",
    "inflect==7.3.1",
    "librosa==0.10.2",
    "lightning==2.2.4",
    "matplotlib==3.7.5",
    "modelscope==1.20.0",
    "networkx==3.1",
    "numpy==1.26.4",
    "pandas==2.2.2",
    "omegaconf==2.3.0",
    "onnx==1.16.0",
    "onnxruntime-gpu==1.18.0",
    # openai-whisper omitted: fails to build on Kaggle Python 3.12.
    # whisper_mel.py + stubs/whisper/ provide whisper.log_mel_spectrogram instead.
    "protobuf==4.25",
    "pyarrow==18.1.0",
    "pydantic==2.7.0",
    "pyworld==0.3.4",
    "rich==13.7.1",
    "soundfile==0.12.1",
    "tensorboard==2.14.0",
    "transformers==4.51.3",
    "tiktoken",
    "x-transformers==2.11.24",
    "wetext==0.0.4",
    "wget==3.2",  # Matcha-TTS (feat_extractor in cosyvoice3 yaml)
    "deepspeed==0.15.1",
]
print(f"[1] installing {len(REQS)} packages (this can take ~3 min) ...")
t0 = time.time()
subprocess.check_call([PYTHON, "-m", "pip", "install", "-q", "--no-cache-dir"] + REQS)
# Kaggle ships pandas built against a different numpy; re-pin both together.
subprocess.check_call([
    PYTHON, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
    "numpy==1.26.4", "pandas==2.2.2",
])
print(f"[1]   done in {time.time()-t0:.1f}s")

try:
    ensure_repo_layout()
except FileNotFoundError as exc:
    print(f"[1]   repo staging skipped ({exc})")

whisper_stub_paths()
import whisper as _whisper_check  # noqa: E402
from whisper.tokenizer import Tokenizer as _WhisperTokenizer  # noqa: E402
print(f"[1] whisper stub ok (log_mel from {_whisper_check.log_mel_spectrogram.__module__})")
print(f"[1] whisper.tokenizer.Tokenizer ok ({_WhisperTokenizer.__module__})")

# Clone CosyVoice (shallow clone keeps it fast)
if not (WORKDIR / "CosyVoice").exists():
    print("[1] cloning CosyVoice (shallow)")
    subprocess.check_call([
        "git", "clone", "--depth=1", "--recursive",
        "https://github.com/FunAudioLLM/CosyVoice.git",
        str(WORKDIR / "CosyVoice"),
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    print("[1] CosyVoice already present")

# Patch: CosyVoice uses torch.pca_lowrank which was removed in newer PyTorch.
# This monkey-patch keeps it working with the deepspeed/torch combo Kaggle ships.
import torch
if not hasattr(torch, "pca_lowrank"):
    def _pca_lowrank(X, q=None, center=True, niter=2):
        if center:
            X = X - X.mean(dim=0, keepdim=True)
        return torch.svd_lowrank(X, q=q or 1, niter=niter)
    torch.pca_lowrank = _pca_lowrank
    print("[1] patched torch.pca_lowrank")

# Make `from cosyvoice.X import Y` work
sys.path.insert(0, str(WORKDIR / "CosyVoice"))
sys.path.insert(0, str(WORKDIR / "CosyVoice" / "third_party" / "Matcha-TTS"))
print(f"[1] cosyvoice on path: {(WORKDIR / 'CosyVoice') in [Path(p) for p in sys.path]}")

# GPU sanity
print(f"[1] torch={torch.__version__}  cuda={torch.cuda.is_available()}  "
      f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")


[1] Python: 3.12.13  executable: /usr/bin/python3
[1] installing 25 packages (this can take ~3 min) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.4/53.4 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 68.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 30.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 36.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.0/252.0 kB 24.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 197.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 208.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 379.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 351.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 263.5 MB/s eta 0:00:00
   ━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.5 requires pyarrow>=21.0.0, but you have pyarrow 18.1.0 which is incompatible.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.29.3 requires pydantic>=2.11.4, but you have pydantic 2.7.0 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
sigstore-models 0.0.6 requires pydantic>=2.12, but you have pydantic 2.7.0 which is incompatible.
google-cloud-videointelligence 2.19.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.0 which is incompatible.
google-cloud-vision 3.14

[1]   done in 172.6s
[1]   repo staging skipped (!! stubs/whisper/__init__.py not found under /kaggle/working. Copy the SinhalaTTS repo files into /kaggle/working/ (same folder as this notebook).)
[1] whisper stub ok (log_mel from whisper_mel)
[1] cloning CosyVoice (shallow)
[1] cosyvoice on path: True
[1] torch=2.10.0+cu128  cuda=True  device=Tesla T4


In [3]:
# =============================================================================
# CELL 2 — Download pretrained model + Sinhala data
# =============================================================================
# We pull:
#   (a) Fun-CosyVoice3-0.5B-2512 from Hugging Face (~12 GB on disk)
#   (b) OpenSLR30 Sinhala TTS from openslr.org/30:
#       si_lk.tar.gz (audio) + si_lk.lines.txt (transcripts, separate download)
#
# Dataset: Google-collected multi-speaker Sinhala corpus (SLR30, CC BY-SA 4.0).
# Layout: si_lk/si_lk.lines.txt + si_lk/wav/*.wav

PRETRAIN_DIR = WORKDIR / "pretrained_models" / "Fun-CosyVoice3-0.5B-2512"
PRETRAIN_DIR.parent.mkdir(parents=True, exist_ok=True)

if not PRETRAIN_DIR.exists() or not (PRETRAIN_DIR / "cosyvoice3.yaml").exists():
    print("[2] downloading Fun-CosyVoice3-0.5B-2512 from HuggingFace ...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="FunAudioLLM/Fun-CosyVoice3-0.5B-2512",
        local_dir=str(PRETRAIN_DIR),
        allow_patterns=[
            "*.json", "*.yaml", "*.pt", "*.onnx",
            "CosyVoice-BlankEN/*", "*.md",
        ],
    )
    print(f"[2]   done -> {PRETRAIN_DIR}")
else:
    print(f"[2] pretrained model already at {PRETRAIN_DIR}")

# OpenSLR30 Sinhala TTS (openslr.org/30)
SINHALA_SRC, _wav_dir = ensure_openslr30_corpus(WORKDIR)



[2] downloading Fun-CosyVoice3-0.5B-2512 from HuggingFace ...


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

vocab.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

...

100  667M  100  667M    0     0  23.9M      0  0:00:27  0:00:27 --:--:-- 25.2M


[2]   2064 sin_*.wav found after extract
[2]   normalizing wav layout: /kaggle/working -> /kaggle/working/si_lk/wav
[2] downloading OpenSLR30 manifest si_lk.lines.txt ...


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  8  193k    8 15934    0     0  24538      0  0:00:08 --:--:--  0:00:08 24513

[2]   corpus -> /kaggle/working/si_lk
[2]   2064 wavs under /kaggle/working/si_lk/wav


100  193k  100  193k    0     0   218k      0 --:--:-- --:--:-- --:--:--  218k


In [4]:
# =============================================================================
# CELL 3 — Data preparation (Kaldi-style files, with Sinhala normalization)
# =============================================================================
# This is the most important step for naturalness. The OpenSLR30 transcripts
# are mostly clean but we run them through our Sinhala text normalizer to
# guarantee consistent unicode encoding (NFC) before the Qwen2 tokenizer
# sees them. See scripts/sinhala_normalize.py for the rules.

# Lay out helper scripts (copied from /kaggle/working/ repo files)
ensure_repo_layout()
sys.path.insert(0, str(SCRIPTS_DIR))

DATA_OUT = WORKDIR / "sinhala_data"
DATA_OUT.mkdir(parents=True, exist_ok=True)

print("[3] running prepare_sinhala_data.py ...")
subprocess.check_call([
    PYTHON, str(SCRIPTS_DIR / "prepare_sinhala_data.py"),
    "--src_dir", str(SINHALA_SRC),
    "--des_dir", str(DATA_OUT),
    "--min_dur", "0.8",
    "--max_dur", "22.0",
    "--dev_speaker_ratio", "0.10",
    "--out_wav_dir", str(DATA_OUT / "wav_24k"),
])
print(f"[3]   summary:")
summary = json.loads((DATA_OUT / "prep_summary.json").read_text())
print(f"[3]   train utts: {summary['train']['utts']}  dev utts: {summary['dev']['utts']}  "
      f"spks: {summary['train']['spks']} / {summary['dev']['spks']}")



[3] running prepare_sinhala_data.py ...
[1/4] reading manifest: /kaggle/working/si_lk/si_lk.lines.txt
      raw rows: 1251
[2/4] filtering + normalizing + resampling


  0%|          | 0/1251 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:178: UserWarning: The 'encoding' parameter is not fully supported by TorchCodec AudioEncoder.
  return save_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:178: UserWarning: The 'bits_per_sample' parameter is not directly supported by TorchCodec AudioEncoder.
  return save_with_torchcodec(
100%|██████████| 1251/1251 [00:25<00:00, 49.91it/s]


      kept: 1251 | skipped: {'no_wav': 0, 'too_short': 0, 'too_long': 0, 'bad_text': 0, 'resample_fail': 0}
[3/4] splitting train/dev (speaker-disjoint, dev_speaker_ratio=0.1)
      train: 1142 utts | dev: 109 utts
[4/4] writing Kaldi-style files
      train: {'utts': 1142, 'spks': 11, 'spk2utt': 1142}
      dev:   {'utts': 109, 'spks': 1, 'spk2utt': 109}
      summary -> /kaggle/working/sinhala_data/prep_summary.json
[3]   summary:
[3]   train utts: 1142  dev utts: 109  spks: 11 / 1


In [12]:
import os
if "__file__" not in globals():
    __file__ = "/kaggle/working/SinhalaTTS/cosyvoice3_sinhala_kaggle.py"

In [5]:
# =============================================================================
# CELL 4 — Feature extraction: speaker embeddings + discrete speech tokens
# =============================================================================
# This is the biggest disk I/O. The CosyVoice3 ONNX files are loaded into
# the model; we run campplus for speaker embeddings (CPU) and
# speech_tokenizer_v3 for the discrete LLM targets (GPU).

ensure_repo_layout()

CAMPPLUS = PRETRAIN_DIR / "campplus.onnx"
SPEECH_TOK = PRETRAIN_DIR / "speech_tokenizer_v3.onnx"
assert CAMPPLUS.exists(), f"!! {CAMPPLUS} missing"
assert SPEECH_TOK.exists(), f"!! {SPEECH_TOK} missing"

for split in ("train", "dev"):
    out_dir = DATA_OUT / split
    print(f"[4] extracting features for {split} ...")
    t0 = time.time()
    subprocess.check_call([
        PYTHON, str(SCRIPTS_DIR / "extract_features.py"),
        "--data_dir", str(out_dir),
        "--campplus_onnx", str(CAMPPLUS),
        "--speech_tokenizer_onnx", str(SPEECH_TOK),
        "--device", "cpu",  # onnxruntime-gpu on Kaggle lacks libcublasLt.so.11
        "--save_every", "200",
    ])
    print(f"[4]   {split} features done in {time.time()-t0:.1f}s")



[4] extracting features for train ...
  loading campplus ONNX on CPU
  loading speech_tokenizer ONNX (requested: cpu)
  speech_tokenizer active provider: CPUExecutionProvider
  to process: 1142 | already done: 0


extract: 100%|██████████| 1142/1142 [18:37<00:00,  1.02it/s]


  flushed @ 200 (231.6s, 1091s left)
  flushed @ 400 (408.5s, 758s left)
  flushed @ 600 (598.7s, 541s left)
  flushed @ 800 (787.9s, 337s left)
  flushed @ 1000 (988.7s, 140s left)
  done. utt2emb=1142 spk2emb=11 utt2tok=1142
[4]   train features done in 1124.1s
[4] extracting features for dev ...
  loading campplus ONNX on CPU
  loading speech_tokenizer ONNX (requested: cpu)
  speech_tokenizer active provider: CPUExecutionProvider
  to process: 109 | already done: 0


extract: 100%|██████████| 109/109 [01:57<00:00,  1.08s/it]


  done. utt2emb=109 spk2emb=1 utt2tok=109
[4]   dev features done in 124.2s


In [6]:
# =============================================================================
# CELL 5 — Build parquet shards
# =============================================================================
ensure_repo_layout()

for split in ("train", "dev"):
    out_dir = DATA_OUT / split
    parquet_dir = out_dir / "parquet"
    parquet_dir.mkdir(exist_ok=True)
    print(f"[5] building parquet for {split} ...")
    subprocess.check_call([
        PYTHON, str(SCRIPTS_DIR / "build_parquet.py"),
        "--data_dir", str(out_dir),
        "--utt2emb", str(out_dir / "utt2embedding.pt"),
        "--spk2emb", str(out_dir / "spk2embedding.pt"),
        "--utt2tok", str(out_dir / "utt2speech_token.pt"),
        "--out_dir", str(parquet_dir),
        "--num_utts_per_parquet", "500",
    ])

# Build concatenated train/dev data.list
(DATA_OUT / "train.data.list").write_text(
    (DATA_OUT / "train/parquet/data.list").read_text(encoding="utf-8")
)
(DATA_OUT / "dev.data.list").write_text(
    (DATA_OUT / "dev/parquet/data.list").read_text(encoding="utf-8")
)
print(f"[5]   wrote {DATA_OUT}/train.data.list and dev.data.list")



[5] building parquet for train ...
  loading Kaldi-style files from /kaggle/working/sinhala_data/train
  1142 utts, 11 spks
  loading pre-extracted features
  utt2emb=1142 spk2emb=11 utt2tok=1142


sharding: 100%|██████████| 3/3 [00:02<00:00,  1.08it/s]


  shard 1: 500 utts -> parquet_000000000.parquet [500/1142  rate=409.5 u/s  eta=2s]
  shard 2: 500 utts -> parquet_000000001.parquet [1000/1142  rate=402.6 u/s  eta=0s]
  shard 3: 142 utts -> parquet_000000002.parquet [1142/1142  rate=409.9 u/s  eta=0s]
  wrote data.list / utt2data.list / spk2data.list -> /kaggle/working/sinhala_data/train/parquet
[5] building parquet for dev ...
  loading Kaldi-style files from /kaggle/working/sinhala_data/dev
  109 utts, 1 spks
  loading pre-extracted features
  utt2emb=109 spk2emb=1 utt2tok=109


sharding: 100%|██████████| 1/1 [00:00<00:00,  3.81it/s]


  shard 1: 109 utts -> parquet_000000000.parquet [109/109  rate=412.8 u/s  eta=0s]
  wrote data.list / utt2data.list / spk2data.list -> /kaggle/working/sinhala_data/dev/parquet
[5]   wrote /kaggle/working/sinhala_data/train.data.list and dev.data.list


In [8]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "--force-reinstall", "numpy==1.26.4", "pandas==2.2.2",
])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 282.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 310.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 321.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 kB 366.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 357.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.5 requires pyarrow>=21.0.0, but you have pyarrow 18.1.0 which is incompatible.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.29.3 requires pydantic>=2.11.4, but you have pydantic 2.7.0 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
mne 1.12.1 requires matplotlib>=3.8, but you have matplotlib 3.7.5 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have ju

0

In [9]:
# =============================================================================
# CELL 6 — Pre-training sanity check
# =============================================================================
# Verify the parquet pipeline works end-to-end BEFORE we commit to a long
# training run. We load one parquet, run the cosyvoice data pipeline, and
# print a summary of the produced tensors. If anything is wrong, this
# surfaces it in 30 seconds.

print("[6] pre-training data sanity check ...")
import pyarrow.parquet as pq

parquet_files = sorted((DATA_OUT / "train/parquet").glob("parquet_*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"no parquet shards in {DATA_OUT / 'train/parquet'}")
sample_pq = parquet_files[0]
table = pq.read_table(sample_pq)
row = table.slice(0, 1)
audio = row.column("audio_data")[0].as_py()
text = row.column("text")[0].as_py()
spk = row.column("spk")[0].as_py()
utt_emb = row.column("utt_embedding")[0].as_py()
speech_tok = row.column("speech_token")[0].as_py()

print(f"[6]   parquet {sample_pq.name} -> {table.num_rows} utts, "
      f"audio bytes: {len(audio)}")
print(f"[6]   sample text: {text!r}")
print(f"[6]   sample spk:  {spk}")
print(f"[6]   sample utt_embedding len: {len(utt_emb)}")
print(f"[6]   sample speech_token len:  {len(speech_tok)}")

import gc
gc.collect()
torch.cuda.empty_cache()
print("[6]   ok!")


[6] pre-training data sanity check ...
[6]   parquet parquet_000000000.parquet -> 500 utts, audio bytes: 303182
[6]   sample text: 'කෝකටත් මං වෙනදා තරම් කාලෙ ගන්නැතිව ඇඳ ගත්තා'
[6]   sample spk:  sin_2241
[6]   sample utt_embedding len: 192
[6]   sample speech_token len:  158
[6]   ok!


In [26]:
# =============================================================================
# CELL 7 — Train LLM (Qwen2-BlankEN, 30 epochs, bf16, deepspeed stage 2)
# =============================================================================
# This is the heart of the SFT. We fine-tune the Qwen2-based LLM on the
# (sinhala text -> speech tokens) mapping. After this step, the model
# knows how to produce Sinhala speech tokens from Sinhala text. The Flow
# + HiFi-GAN below further refine the audio quality, but the LLM is the
# biggest factor for *naturalness* of prosody.
#
# Expected time on T4: ~4-5 hours for 30 epochs over ~1.5k utts.

ensure_repo_layout()
bootstrap_whisper_stub()

def patch_constantlr_scheduler_conf(cfg_path: Path) -> None:
    """Make constantlr + DeepSpeed safe: CosyVoice ConstantLR accepts no kwargs."""
    import re

    text = cfg_path.read_text(encoding="utf-8")
    if "constantlr" not in text:
        return
    text = re.sub(r"^[ \t]*warmup_steps:.*(?:\n|$)", "", text, flags=re.M)
    text = re.sub(
        r"^([ \t]*scheduler_conf:\s*)\n"
        r"(?=[ \t]*(?:max_epoch|grad_clip|accum_grad|log_interval|save_per_step):)",
        r"\1 {}\n",
        text,
        flags=re.M,
    )
    cfg_path.write_text(text, encoding="utf-8")


def patch_cosyvoice_pytorch_compat(repo_root: Path) -> None:
    """Fix cosyvoice_join on PyTorch 2.x (ProcessGroup.options removed) and skip on 1 GPU."""
    path = repo_root / "cosyvoice" / "utils" / "train_utils.py"
    if not path.exists():
        return
    text = path.read_text(encoding="utf-8")
    changed = False
    if "group_join.options._timeout" in text:
        text = text.replace(
            "group_join.options._timeout",
            "(getattr(getattr(group_join, 'options', None), '_timeout', None) "
            "or datetime.timedelta(seconds=int(os.environ.get('COSYVOICE_JOIN_TIMEOUT', '60'))))",
        )
        changed = True
    join_block = 'rank = int(os.environ.get(\'RANK\', 0))'
    if join_block in text and "if world_size <= 1:" not in text.split("def cosyvoice_join", 1)[1].split("def batch_forward", 1)[0]:
        text = text.replace(
            join_block + '\n\n    if info_dict["batch_idx"]',
            join_block + "\n\n    if world_size <= 1:\n        return False\n\n    if info_dict[\"batch_idx\"]",
            1,
        )
        changed = True
    if changed:
        path.write_text(text, encoding="utf-8")


TARGET_CONFIG = WORKDIR / "CosyVoice" / "examples" / "libritts" / "cosyvoice3" / "conf" / "cosyvoice3_sinhala_sft.yaml"
TARGET_DS = WORKDIR / "CosyVoice" / "examples" / "libritts" / "cosyvoice3" / "conf" / "ds_stage2.json"
TARGET_CONFIG.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(CONFIGS_DIR / "cosyvoice3_sinhala_sft.yaml", TARGET_CONFIG)
shutil.copy(CONFIGS_DIR / "ds_stage2.json", TARGET_DS)
patch_constantlr_scheduler_conf(TARGET_CONFIG)
if "warmup_steps" in TARGET_CONFIG.read_text(encoding="utf-8"):
    raise RuntimeError(
        "[7] scheduler_conf still has warmup_steps after patch — "
        "check cosyvoice3_sinhala_sft.yaml"
    )
print("[7] scheduler_conf patched for constantlr + DeepSpeed")
patch_cosyvoice_pytorch_compat(WORKDIR / "CosyVoice")
print("[7] CosyVoice train_utils patched for PyTorch 2.x / single GPU")

launcher_src = SCRIPTS_DIR / "train_sinhala_sft.sh"

# Matcha-TTS imports wget when loading the yaml feat_extractor (skip if CELL 1 ran).
subprocess.check_call([PYTHON, "-m", "pip", "install", "-q", "wget==3.2"])

# Run training
print("[7] launching LLM training ...")
_repo = WORKDIR / "CosyVoice"
_matcha = _repo / "third_party" / "Matcha-TTS"
_py_path = os.pathsep.join([
    str(_repo), str(_matcha), str(SCRIPTS_DIR), str(STUBS_DIR),
])
env = os.environ.copy()
env.update({
    "REPO_ROOT": str(_repo),
    "PRETRAINED_DIR": str(PRETRAIN_DIR),
    "DATA_DIR": str(DATA_OUT),
    "EXP_DIR": str(WORKDIR / "exp" / "cosyvoice3"),
    "TB_DIR": str(WORKDIR / "tensorboard" / "cosyvoice3"),
    "CONFIG": str(TARGET_CONFIG),
    "DS_CONFIG": str(TARGET_DS),
    "SCRIPTS_DIR": str(SCRIPTS_DIR),
    "STUBS_DIR": str(STUBS_DIR),
    "PYTHONPATH": _py_path,
    "NUM_WORKERS": "2",
    "PREFETCH": "100",
    "SAVE_EVERY": "500",
    "LOG_INTERVAL": "50",
    "MAX_EPOCH": "30",
    "AVERAGE_NUM": "5",
    "CUDA_VISIBLE_DEVICES": "0",
})
subprocess.check_call(["bash", str(launcher_src), "llm"], env=env)
print("[7]   LLM training complete")

gc.collect()
torch.cuda.empty_cache()

[7] scheduler_conf patched for constantlr + DeepSpeed
[7] CosyVoice train_utils patched for PyTorch 2.x / single GPU
[7] launching LLM training ...
  TRAINING MODEL: llm
  pretrained:     /kaggle/working/pretrained_models/Fun-CosyVoice3-0.5B-2512/llm.pt
  output:         /kaggle/working/exp/cosyvoice3/llm/deepspeed
  tensorboard:    /kaggle/working/tensorboard/cosyvoice3/llm/deepspeed


[W620 09:07:27.904396745 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


[2026-06-20 09:07:30,004] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


2026-06-20 09:07:35.155202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781946455.179667    1169 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781946455.187711    1169 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781946455.208059    1169 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781946455.208105    1169 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781946455.208111    1169 computation_placer.cc:177] computation placer alr

[2026-06-20 09:07:44,292] [INFO] [comm.py:652:init_distributed] cdb=None
[2026-06-20 09:07:44,292] [INFO] [comm.py:683:init_distributed] Initializing TorchBackend in DeepSpeed with backend nccl


...

[2026-06-20 09:07:52,371] [INFO] [torch_checkpoint_engine.py:23:save] [Torch] Saved /kaggle/working/exp/cosyvoice3/llm/deepspeed/init/mp_rank_00_model_states.pt.
[2026-06-20 09:07:52,372] [INFO] [torch_checkpoint_engine.py:21:save] [Torch] Saving /kaggle/working/exp/cosyvoice3/llm/deepspeed/init/bf16_zero_pp_rank_0_mp_rank_00_optim_states.pt...
[2026-06-20 09:07:57,811] [INFO] [torch_checkpoint_engine.py:23:save] [Torch] Saved /kaggle/working/exp/cosyvoice3/llm/deepspeed/init/bf16_zero_pp_rank_0_mp_rank_00_optim_states.pt.
[2026-06-20 09:07:57,812] [INFO] [engine.py:3536:_save_zero_checkpoint] zero checkpoint saved /kaggle/working/exp/cosyvoice3/llm/deepspeed/init/bf16_zero_pp_rank_0_mp_rank_00_optim_states.pt
[2026-06-20 09:07:57,812] [INFO] [torch_checkpoint_engine.py:33:commit] [Torch] Checkpoint init is ready now!
start step 0 start epoch -1
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


/kaggle/working/CosyVoice/cosyvoice/bin/train.py:177: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if args.use_amp else None
/kaggle/working/CosyVoice/cosyvoice/utils/train_utils.py:255: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  autocast = torch.cuda.amp.autocast(enabled=True, dtype=dtype, cache_enabled=False)


[2026-06-20 09:10:27,687] [INFO] [logging.py:96:log_dist] [Rank 0] step=50, skipped=0, lr=[1e-05], mom=[(0.9, 0.95)]
[2026-06-20 09:10:27,782] [INFO] [timer.py:259:stop] epoch=0/micro_step=50/global_step=50, RunningAvgSamplesPerSec=0.5675045599578646, CurrSamplesPerSec=0.5544797557851442, MemAllocated=6.62GB, MaxMemAllocated=11.03GB
[2026-06-20 09:11:55,764] [INFO] [logging.py:96:log_dist] [Rank 0] step=100, skipped=0, lr=[1e-05], mom=[(0.9, 0.95)]
[2026-06-20 09:11:55,860] [INFO] [timer.py:259:stop] epoch=0/micro_step=100/global_step=100, RunningAvgSamplesPerSec=0.5678732003360036, CurrSamplesPerSec=0.6233346999261496, MemAllocated=6.62GB, MaxMemAllocated=11.07GB
[2026-06-20 09:13:22,247] [INFO] [logging.py:96:log_dist] [Rank 0] step=150, skipped=0, lr=[1e-05], mom=[(0.9, 0.95)]
[2026-06-20 09:13:22,342] [INFO] [timer.py:259:stop] epoch=0/micro_step=150/global_step=150, RunningAvgSamplesPerSec=0.5714781136598862, CurrSamplesPerSec=0.5292465643629969, MemAllocated=6.62GB, MaxMemAllocat

[rank0]: Traceback (most recent call last):
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 977, in save
[rank0]:     _save(
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1286, in _save
[rank0]:     zip_file.write_record(name, storage, num_bytes)
[rank0]: RuntimeError: basic_ios::clear: iostream error

[rank0]: During handling of the above exception, another exception occurred:

[rank0]: Traceback (most recent call last):
[rank0]:   File "/kaggle/working/CosyVoice/cosyvoice/bin/train.py", line 195, in <module>
[rank0]:     main()
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 362, in wrapper
[rank0]:     return f(*args, **kwargs)
[rank0]:            ^^^^^^^^^^^^^^^^^^
[rank0]:   File "/kaggle/working/CosyVoice/cosyvoice/bin/train.py", line 190, in main
[rank0]:     executor.train_one_epoc(model, optimizer, scheduler, train_data_lo

CalledProcessError: Command '['bash', '/kaggle/working/scripts/train_sinhala_sft.sh', 'llm']' returned non-zero exit status 1.

In [25]:
import os
from pathlib import Path

path = Path("/kaggle/working/CosyVoice/cosyvoice/utils/train_utils.py")
text = path.read_text(encoding="utf-8")

if "group_join.options._timeout" in text:
    text = text.replace(
        "group_join.options._timeout",
        "(getattr(getattr(group_join, 'options', None), '_timeout', None) "
        "or __import__('datetime').timedelta(seconds=int(os.environ.get('COSYVOICE_JOIN_TIMEOUT', '60'))))",
    )

anchor = "rank = int(os.environ.get('RANK', 0))"
if anchor in text and "if world_size <= 1:" not in text.split("def cosyvoice_join", 1)[1][:400]:
    text = text.replace(
        anchor + '\n\n    if info_dict["batch_idx"]',
        anchor + "\n\n    if world_size <= 1:\n        return False\n\n    if info_dict[\"batch_idx\"]",
        1,
    )

path.write_text(text, encoding="utf-8")
print("patched", path)

patched /kaggle/working/CosyVoice/cosyvoice/utils/train_utils.py


In [22]:
!grep -A2 "scheduler:" /kaggle/working/CosyVoice/examples/libritts/cosyvoice3/conf/cosyvoice3_sinhala_sft.yaml

                t_scheduler: 'cosine'
                training_cfg_rate: 0.2
                inference_cfg_rate: 0.7
--
    scheduler: constantlr
    scheduler_conf: {}
    max_epoch: 30
--
    scheduler: constantlr
    optim_d: adam
    optim_conf_d:
